### Задание 1 (наследование и перегрузка операторов)
Требуется написать класс-строку: такой, чтобы его инстансы можно было сравнивать между собой. При этом, приоритетом для того чтобы считать строку большей, будет считаться длина строки, уже во вторую очередь будет сравниваться лексикографическая составляющая. В остальном класс должен повторять возможности обычных строк

In [ ]:
'ac' > 'ab'  # лексикографическое сравнение 

In [ ]:
'ac' > 'aba'  # сравнение по длинам которое нас не устраивает 

In [ ]:
'ac'.upper()

In [12]:
class Str(str):
    def __lt__(self, other):
        if len(self) < len(other):
            return True
        return super().__lt__(other)
    
    def __gt__(self, other):
        if len(self) > len(other):
            return True
        return super().__gt__(other)
        
    
Str('ac') > Str('ab')

True

In [13]:
Str('ac') < Str('a')

False

In [14]:
Str('ac') > Str('a')

True

In [15]:
Str('ac') < Str('aba')

True

In [16]:
Str('ac') > Str('aba')

True

In [17]:
Str('ac').upper()

'AC'

### Задание 2 (полиморфизм и инкапсуляция)
Напишите класс для каждой из фигур:
`
- Прямоугольник `Rectangle`. Принимает в конструктор стороны `a` и `b`.
- Круг `Circle`. Принимает в конструктор радиус `r`.
- Ромб `Rhombus`. Принимает в конструктор диагонали `p` и `q`.

Реализуйте в каждом из классов методы для вычисления периметра `get_perimeter` и площади `get_square`. Вам не требуется округлять значения, возвращаемые методами до 2х знаков после запятой, тест уже это делает.

Существующие функции `calculate_perimeter(figure)` и `calculate_square(figure)` должны иметь возможность работать с экземплярами каждой из фигур.

In [30]:
import math

class Rectangle:
    def __init__(self, a, b):
        self.a = a
        self.b = b
    
    def get_perimeter(self):
        return (self.a + self.b) * 2
    
    def get_square(self):
        return self.a * self.b
    

class Circle:
    def __init__(self, r):
        self.r = r
    
    def get_perimeter(self):
        return self.r**2 * math.pi
    
    def get_square(self):
        return 2 * self.r * math.pi
    

class Rhombus:
    def __init__(self, p, q):
        self.p = p
        self.q = q
    
    def get_perimeter(self):
        return 2 * math.sqrt(self.p**2 + self.q**2)

    def get_square(self):
        return self.p * self.q / 2
    

def calculate_perimeter(figure):
    return figure.get_perimeter()

def calculate_square(figure):
    return figure.get_square()

rect = Rectangle(1, 2)
circ = Circle(2)
rhomb = Rhombus(1, 2)

print(*[calculate_perimeter(figure) for figure in [rect, circ, rhomb]])
print(*[calculate_square(figure) for figure in [rect, circ, rhomb]])

6 12.566370614359172 4.47213595499958
2 12.566370614359172 1.0


### Задание 3 (наследование)
В модуле `collections` есть класс `Counter`, который может подсчитать число элементов в любой последовательности, в том числе в строках. Этот класс имеет метод `most_common`, который принимает на вход один необязательный аргумент `n` и возвращает `n` самых частых элементов, если `n` не указано, возвращает все элементы в порядке сортировки от самых частых к самым редковстречающимся. Порядок элементов с одной частотой не имеет значения

Реализуйте на основе `Counter` класс `MyCounter`, определите в нём метод `least_common`, который будет принимать на вход один необязательный аргумент `n` и возвращать `n` самых редких элементов, если `n` не указано, будет возвращать все элементы в порядке сортировки от самых редких к самым частовстречающимся. В остальном `MyCounter` должен быть подобен `Counter`

In [48]:
from collections import Counter

class MyCounter(Counter):
    def least_common(self, n=None):
        return self.most_common()[::-1][:n]


c = MyCounter("abracadabra")

print(c.least_common())
print(c.least_common(3))

[('d', 1), ('c', 1), ('r', 2), ('b', 2), ('a', 5)]
[('d', 1), ('c', 1), ('r', 2)]


### Задание 4 (абстракция)
Класс - банковский вклад. 
Составляющие:
- процент
- сумм
- период
- капитализация
- методы управления

Нужно описать класс

In [54]:
# задача не так проста, как может показаться. Для начала, необходимо понять предметную область...
# 1) банковские вклады могут быть с возможностью пополнения / без возможности пополнения
# 2) с автоматическим продлением / без
# 3) с возможностью частичного снятия денежных средств / без (можно закрыть только весь вклад, при этом проценты не будут начислены)
# 4) капитализация также будет являться логическим признаком, она может быть / её может не быть

# предметная область задаёт некоторые огранияения на сущность "вклад"
# очерчиваются возможные методы:
# - открыть / продлить / закрыть
# - пополнить / снять
# - подождать, при этом важно учитывать факторы 1-4
# - рассчитать на некоторый срок

# Также необходимо задать некоторые допущения:
# - проценты начисляются (вычисляются, если вклад без частичного снятия и/или преждевременного закрытыия) каждый месяц
# - процент указывается за месяц, а не весь период вклада ... если годовой процент = 6, то месячный будем считать 0.5
# - период указывается в месяцах

class Deposit:
    def __init__(
        self,
        percent: float, # месячный процент
        amount: float,  # начальная сумма
        period: int,    # срок в месяцах

        can_replenish: bool,     # можно пополнять
        can_withdraw: bool,      # можно снять
        auto_prolongation: bool, # можно автопродливать
        capitalization: bool     # может быть капитализация
    ):
        # параметры продукта
        self.percent = percent / 100
        self.initial_amount = amount
        self.period = period

        self.can_replenish = can_replenish
        self.can_withdraw = can_withdraw
        self.auto_prolongation = auto_prolongation
        self.capitalization = capitalization

        # текущее состояние
        self.balance = amount
        self.accrued_interest = 0.0
        self.current_month = 0
        self.is_open = False

    def open(self):
        if self.is_open:
            raise Exception("Вклад уже открыт")
        self.is_open = True

    def close(self):
        if not self.is_open:
            raise Exception("Вклад не открыт")

        self.is_open = False

        if not self.can_withdraw:
            return self.balance

        return self.balance + self.accrued_interest

    def replenish(self, amount: float):
        if not self.can_replenish:
            raise Exception("Пополнение запрещено")

        if not self.is_open:
            raise Exception("Вклад не открыт")

        self.balance += amount

    def withdraw(self, amount: float):
        if not self.can_withdraw:
            raise Exception("Частичное снятие запрещено")

        if amount > self.balance:
            raise Exception("Недостаточно средств")

        self.balance -= amount

    def wait(self):
        if not self.is_open:
            return

        self.current_month += 1

        interest = self.balance * self.percent

        if self.capitalization:
            self.balance += interest
        else:
            self.accrued_interest += interest

        if self.current_month >= self.period:
            if self.auto_prolongation:
                self.current_month = 0
            else:
                self.close()
    
    def simulate(self, months: int):
        balance = self.balance
        accrued = self.accrued_interest

        for _ in range(months):
            interest = balance * self.percent

            if self.capitalization:
                balance += interest
            else:
                accrued += interest

        return balance, accrued

In [ ]:
# Вклад с капитализацией и пополнением
deposit = Deposit(
    percent=0.5,
    amount=1000,
    period=6,

    can_replenish=True,
    can_withdraw=False,
    auto_prolongation=False,
    capitalization=True
)

deposit.open()

deposit.replenish(500)

for _ in range(6):
    deposit.wait()

print("Баланс:", deposit.balance)
print("Проценты:", deposit.accrued_interest)

Баланс: 1545.5662640906485
Проценты: 0.0


In [60]:
# Без капитализации, с частичным снятием
deposit = Deposit(
    percent=0.5,
    amount=2000,
    period=6,

    can_replenish=False,
    can_withdraw=True,
    auto_prolongation=False,
    capitalization=False
)

deposit.open()

deposit.wait()
deposit.wait()

deposit.withdraw(500)

deposit.wait()
deposit.wait()

print("Баланс:", deposit.balance)
print("Начисленные проценты:", deposit.accrued_interest)

print("Итог при закрытии:", deposit.close())

Баланс: 1500
Начисленные проценты: 35.0
Итог при закрытии: 1535.0


In [ ]:
# Автопролонгация
deposit = Deposit(
    percent=0.5,
    amount=1000,
    period=3,

    can_replenish=False,
    can_withdraw=False,
    auto_prolongation=True,
    capitalization=True
)

deposit.open()

for i in range(7):
    deposit.wait()
    print(f"Месяц {i+1}: баланс = {deposit.balance}")

Месяц 1: баланс = 1005.0
Месяц 2: баланс = 1010.025
Месяц 3: баланс = 1015.075125
Месяц 4: баланс = 1020.150500625
Месяц 5: баланс = 1025.251253128125
Месяц 6: баланс = 1030.3775093937656
Месяц 7: баланс = 1035.5293969407344


In [61]:
# Прогноз
deposit = Deposit(
    percent=0.5,
    amount=1000,
    period=12,

    can_replenish=False,
    can_withdraw=False,
    auto_prolongation=False,
    capitalization=True
)

deposit.open()

future_balance, future_interest = deposit.simulate(12)

print("Прогноз через 12 месяцев:")
print("Баланс:", future_balance)
print("Проценты:", future_interest)

Прогноз через 12 месяцев:
Баланс: 1061.6778118644995
Проценты: 0.0
